In [56]:
import argparse
import json
from pathlib import Path
from typing import Iterable, List

import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
def read_csv_robust(path: Path, encodings: Iterable[str] = ("utf-8", "latin-1")) -> pd.DataFrame:
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to read CSV {path} ({last_err})")

def build_texts(df: pd.DataFrame, cols: List[str], joiner: str) -> pd.Series:
    parts = [df[c].fillna("").astype(str).str.strip() if c in df.columns
             else pd.Series([""] * len(df), index=df.index) for c in cols]
    s = parts[0]
    for p in parts[1:]:
        s = s + joiner + p
    # collapse multiple joiners caused by empties, and strip
    j = joiner.strip()
    if j:
        s = s.str.replace(rf"(?:\s*{j}\s*)+", f" {j} ", regex=True)
    return s.str.strip()


def process_one_file(
    model: SentenceTransformer,
    in_csv: Path,
    out_csv: Path,
    cols: List[str],
    joiner: str,
    batch_size: int,
    normalize: bool,
):
    df = read_csv_robust(in_csv)

    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        print(f"[WARN] {in_csv.name}: missing columns {missing_cols}; they will be treated as empty strings.")

    sbert_text = build_texts(df, cols, joiner)

    embeddings = model.encode(
        sbert_text.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    )

    out_df = df.copy()
    out_df["sbert_text"] = sbert_text
    out_df["embedding_json"] = [json.dumps(vec.tolist()) for vec in embeddings]

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_csv, index=False)
    print(f"[OK] {in_csv.name} -> {out_csv.name} (rows: {len(out_df)})")

In [8]:
from pathlib import Path

In [57]:
test = pd.read_csv('/data/elugos/sbert_title_121225/20250601.export_goosed_sbert.csv')


In [58]:
test.head()

,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,Actor2CountryCode,...,ActionGeo_FeatureID,DATEADDED,SOURCEURL,domain,target,title,text,description,sbert_text,embedding_json
0,1247164332,20250525,2025,BUS,COMPANY,NaN,NaN,CVL,PROPERTY OWNER,NaN,...,CO,20250601,https://www.greeleytribune.com/2025/05/31/some...,www.greeleytribune.com,greeleytribune,Some Galeton residents still evacuated after B...,Four of 14 Galeton homeowners affected by an o...,There was a failure of the designed well barri...,Some Galeton residents still evacuated after B...,"[-0.017203742638230324, -0.03149743005633354, ..."
1,1247164333,20250525,2025,BUS,COMPANY,NaN,NaN,CVL,PROPERTY OWNER,NaN,...,CO,20250601,https://www.greeleytribune.com/2025/05/31/some...,www.greeleytribune.com,greeleytribune,Some Galeton residents still evacuated after B...,Four of 14 Galeton homeowners affected by an o...,There was a failure of the designed well barri...,Some Galeton residents still evacuated after B...,"[-0.017203742638230324, -0.03149743005633354, ..."
2,1247164334,20250525,2025,CVL,TENANTS,NaN,NaN,NaN,NaN,NaN,...,CO,20250601,https://www.greeleytribune.com/2025/05/31/some...,www.greeleytribune.com,greeleytribune,Some Galeton residents still evacuated after B...,Four of 14 Galeton homeowners affected by an o...,There was a failure of the designed well barri...,Some Galeton residents still evacuated after B...,"[-0.017203742638230324, -0.03149743005633354, ..."
3,1247164335,20250525,2025,CVL,PROPERTY OWNER,NaN,NaN,BUS,COMPANY,NaN,...,CO,20250601,https://www.greeleytribune.com/2025/05/31/some...,www.greeleytribune.com,greeleytribune,Some Galeton residents still evacuated after B...,Four of 14 Galeton homeowners affected by an o...,There was a failure of the designed well barri...,Some Galeton residents still evacuated after B...,"[-0.017203742638230324, -0.03149743005633354, ..."
4,1247164432,20250601,2025,NaN,NaN,NaN,NaN,USA,UNITED STATES,USA,...,CO,20250601,https://www.yardbarker.com/mlb/articles/mets_g...,www.yardbarker.com,yardbarker,"Mets grab early lead, hand Rockies 7th straigh...",Brett Baty hit a bases-clearing triple in a fo...,Brett Baty hit a bases-clearing triple in a fo...,"Mets grab early lead, hand Rockies 7th straigh...","[-0.047115352004766464, 0.07374498248100281, 0..."
